# Bureau Balance Data Cleaning

This notebook cleans the bureau-balance records using the problems found during EDA. It keeps only the monthly records linked to bureau accounts already in the cleaned bureau data, checks the status codes and month values, and creates a few status indicators.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
raw_path = project_root / "data" / "raw" / "bureau_balance.csv"
bureau_path = project_root / "data" / "interim" / "bureau_clean.pkl"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "interim" / "bureau_balance_clean.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [raw_path, bureau_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Raw input:", raw_path)
print("Bureau-account mapping:", bureau_path)
print("Clean output:", output_path)

Raw input: /Users/taranveersingh/A-MRP/data/raw/bureau_balance.csv
Bureau-account mapping: /Users/taranveersingh/A-MRP/data/interim/bureau_clean.pkl
Clean output: /Users/taranveersingh/A-MRP/data/interim/bureau_balance_clean.pkl


## Load bureau mapping and monthly balance data


In [7]:
bureau_mapping = pd.read_pickle(bureau_path)[["SK_ID_BUREAU", "SK_ID_CURR"]]
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
project_ids = training_id_set.union(set(test_ids))
assert bureau_mapping["SK_ID_BUREAU"].is_unique
assert set(bureau_mapping["SK_ID_CURR"]).issubset(project_ids)

balance_raw = pd.read_csv(raw_path)
original_rows, original_columns = balance_raw.shape
balance_clean = balance_raw.merge(
    bureau_mapping, on="SK_ID_BUREAU", how="inner", validate="many_to_one"
).reset_index(drop=True)
del balance_raw
out_of_scope_rows = original_rows - len(balance_clean)
print("Raw monthly rows:", original_rows)
print("Project monthly rows retained:", len(balance_clean))
print("Rows without a project bureau mapping removed:", out_of_scope_rows)
print("Bureau accounts with monthly history:", balance_clean["SK_ID_BUREAU"].nunique())
print("Applicants represented:", balance_clean["SK_ID_CURR"].nunique())

Raw monthly rows: 27299925
Project monthly rows retained: 14701612
Rows without a project bureau mapping removed: 12598313
Bureau accounts with monthly history: 523515
Applicants represented: 92231


Only rows that match a bureau account from the cleaned bureau data are kept, since the rest can't be linked to a project applicant anyway. Most of the raw rows were actually removed here, since the raw bureau-balance file covers a lot more accounts than are in this project.


## Validate identifiers and monthly keys


In [10]:
missing_bureau_ids = int(balance_clean["SK_ID_BUREAU"].isna().sum())
missing_current_ids = int(balance_clean["SK_ID_CURR"].isna().sum())
exact_duplicates = int(balance_clean.duplicated().sum())
duplicate_month_keys = int(balance_clean.duplicated(["SK_ID_BUREAU", "MONTHS_BALANCE"]).sum())
if exact_duplicates > 0:
    balance_clean = balance_clean.drop_duplicates().reset_index(drop=True)
assert missing_bureau_ids == 0 and missing_current_ids == 0
assert duplicate_month_keys == exact_duplicates, "Conflicting records exist for the same bureau account and month."
print("Missing bureau IDs:", missing_bureau_ids)
print("Missing applicant IDs:", missing_current_ids)
print("Exact duplicate rows removed:", exact_duplicates)
print("Conflicting account-month records:", duplicate_month_keys - exact_duplicates)

Missing bureau IDs: 0
Missing applicant IDs: 0
Exact duplicate rows removed: 0
Conflicting account-month records: 0


The IDs are clean, no missing values, no duplicates, no conflicting records for the same account-month.


## Standardize and validate status codes


In [13]:
balance_clean["STATUS"] = balance_clean["STATUS"].astype("string").str.strip()
valid_statuses = {"0", "1", "2", "3", "4", "5", "C", "X"}
invalid_status = ~balance_clean["STATUS"].isin(valid_statuses) | balance_clean["STATUS"].isna()
balance_clean["BB_INVALID_STATUS"] = invalid_status.astype("int8")
balance_clean.loc[invalid_status, "STATUS"] = "X"

status_counts = balance_clean["STATUS"].value_counts().reindex(
    ["0", "1", "2", "3", "4", "5", "C", "X"], fill_value=0
)
print("Invalid or missing status values converted to X:", int(invalid_status.sum()))
status_counts

Invalid or missing status values converted to X: 0


STATUS
0    4615684
1     155330
2      15583
3       5976
4       3897
5      40528
C    7027575
X    2837039
Name: count, dtype: int64[pyarrow]

No invalid or missing status values were found, so nothing actually needed correcting here.


## Validate month values and create status indicators


In [16]:
future_month = balance_clean["MONTHS_BALANCE"].gt(0)
extreme_month = balance_clean["MONTHS_BALANCE"].lt(-1200)
balance_clean["BB_MONTH_ANOMALY"] = (future_month | extreme_month).astype("int8")
balance_clean.loc[future_month | extreme_month, "MONTHS_BALANCE"] = np.nan

balance_clean["BB_STATUS_UNKNOWN"] = balance_clean["STATUS"].eq("X").astype("int8")
balance_clean["BB_STATUS_CLOSED"] = balance_clean["STATUS"].eq("C").astype("int8")
balance_clean["BB_STATUS_DELINQUENT"] = balance_clean["STATUS"].isin(["1", "2", "3", "4", "5"]).astype("int8")
severity_map = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "C": 0}
balance_clean["BB_DELINQUENCY_LEVEL"] = balance_clean["STATUS"].map(severity_map).astype("float32")
print("Future month values corrected:", int(future_month.sum()))
print("Extreme month values corrected:", int(extreme_month.sum()))
print("Delinquent monthly records:", int(balance_clean["BB_STATUS_DELINQUENT"].sum()))
print("Unknown-status monthly records:", int(balance_clean["BB_STATUS_UNKNOWN"].sum()))

Future month values corrected: 0
Extreme month values corrected: 0
Delinquent monthly records: 221314
Unknown-status monthly records: 2837039


No future or extremely old month values were found either. About 221,314 monthly records are flagged as delinquent, and a larger share are flagged as unknown status.


## Build training-only feature decisions


In [19]:
MISSING_THRESHOLD = 0.50
training_balance = balance_clean.loc[balance_clean["SK_ID_CURR"].isin(training_id_set)]
decision_rows = []
for column in balance_clean.columns:
    if column in ["SK_ID_CURR", "SK_ID_BUREAU"]:
        continue
    series = training_balance[column]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    decision = "Keep"
    reason = "Retain for bureau and applicant-level aggregation"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-linked missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant among training-linked monthly records"
    decision_rows.append({
        "feature": column, "data_type": str(series.dtype),
        "missing_count": int(series.isna().sum()), "missing_rate": missing_rate,
        "unique_non_missing": int(unique_non_missing), "decision": decision,
        "reason": reason, "target_association_stage": "After applicant-level aggregation"
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "missing_rate"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[
    feature_decisions["decision"] == "Remove", "feature"
] .tolist()
balance_clean = balance_clean.drop(columns=removed_features)
print("Features removed:", removed_features)
feature_decisions.round(5)

Features removed: ['BB_INVALID_STATUS', 'BB_MONTH_ANOMALY']


,feature,data_type,missing_count,missing_rate,unique_non_missing,decision,reason,target_association_stage
0,BB_DELINQUENCY_LEVEL,float32,2261962,0.19254,6,Keep,Retain for bureau and applicant-level aggregation,After applicant-level aggregation
1,MONTHS_BALANCE,float64,0,0.00000,97,Keep,Retain for bureau and applicant-level aggregation,After applicant-level aggregation
2,STATUS,string,0,0.00000,8,Keep,Retain for bureau and applicant-level aggregation,After applicant-level aggregation
3,BB_STATUS_UNKNOWN,int8,0,0.00000,2,Keep,Retain for bureau and applicant-level aggregation,After applicant-level aggregation
4,BB_STATUS_CLOSED,int8,0,0.00000,2,Keep,Retain for bureau and applicant-level aggregation,After applicant-level aggregation
5,BB_STATUS_DELINQUENT,int8,0,0.00000,2,Keep,Retain for bureau and applicant-level aggregation,After applicant-level aggregation
6,BB_INVALID_STATUS,int8,0,0.00000,1,Remove,Constant among training-linked monthly records,After applicant-level aggregation
7,BB_MONTH_ANOMALY,int8,0,0.00000,1,Remove,Constant among training-linked monthly records,After applicant-level aggregation


Two features were removed: the invalid-status and month-anomaly flags created earlier in this notebook. Since no invalid statuses or bad month values were actually found, both ended up constant.


## Create record-level missingness features


In [22]:
record_features = [c for c in balance_clean.columns if c not in ["SK_ID_CURR", "SK_ID_BUREAU"]]
balance_clean["BB_RECORD_MISSING_COUNT"] = balance_clean[record_features].isna().sum(axis=1).astype("int8")
balance_clean["BB_RECORD_MISSING_RATE"] = balance_clean["BB_RECORD_MISSING_COUNT"] / len(record_features)
balance_clean["STATUS"] = balance_clean["STATUS"].astype("category")
print(balance_clean[["BB_RECORD_MISSING_COUNT", "BB_RECORD_MISSING_RATE"]].describe().round(5))

       BB_RECORD_MISSING_COUNT  BB_RECORD_MISSING_RATE
count             1.470161e+07            1.470161e+07
mean              1.929700e-01            3.216000e-02
std               3.946300e-01            6.577000e-02
min               0.000000e+00            0.000000e+00
25%               0.000000e+00            0.000000e+00
50%               0.000000e+00            0.000000e+00
75%               0.000000e+00            0.000000e+00
max               1.000000e+00            1.666700e-01


This column tracks how much information is missing for each monthly record. The missing rate is low, around 3% on average.


## Validate the cleaned table


In [25]:
numeric_columns = balance_clean.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(balance_clean[c].dropna()).sum()) for c in numeric_columns)
validation_checks = pd.DataFrame([
    {"check": "Only project applicants retained", "passed": set(balance_clean["SK_ID_CURR"]).issubset(project_ids)},
    {"check": "Only cleaned bureau accounts retained", "passed": set(balance_clean["SK_ID_BUREAU"]).issubset(set(bureau_mapping["SK_ID_BUREAU"]))},
    {"check": "Bureau IDs complete", "passed": balance_clean["SK_ID_BUREAU"].notna().all()},
    {"check": "Applicant IDs complete", "passed": balance_clean["SK_ID_CURR"].notna().all()},
    {"check": "Account-month keys unique", "passed": not balance_clean.duplicated(["SK_ID_BUREAU", "MONTHS_BALANCE"]).any()},
    {"check": "No invalid status labels", "passed": set(balance_clean["STATUS"].dropna().astype(str)).issubset(valid_statuses)},
    {"check": "No future balance months", "passed": not balance_clean["MONTHS_BALANCE"].gt(0).any()},
    {"check": "Unknown status has missing severity", "passed": balance_clean.loc[balance_clean["STATUS"].astype(str).eq("X"), "BB_DELINQUENCY_LEVEL"].isna().all()},
    {"check": "No high-missing retained feature", "passed": not (feature_decisions.query("decision == 'Keep'")["missing_rate"] >= MISSING_THRESHOLD).any()},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
])
assert validation_checks["passed"].all(), "At least one bureau-balance cleaning check failed."
validation_checks

,check,passed
0,Only project applicants retained,True
1,Only cleaned bureau accounts retained,True
2,Bureau IDs complete,True
3,Applicant IDs complete,True
4,Account-month keys unique,True
5,No invalid status labels,True
6,No future balance months,True
7,Unknown status has missing severity,True
8,No high-missing retained feature,True
9,No infinite numerical values,True


All checks passed.


## Save the clean table and audit reports


In [28]:
cleaning_audit = pd.DataFrame([
    {"rule": "Rows without project bureau mapping removed", "affected": out_of_scope_rows},
    {"rule": "Exact duplicate rows removed", "affected": exact_duplicates},
    {"rule": "Invalid statuses converted to X", "affected": int(invalid_status.sum())},
    {"rule": "Delinquent monthly statuses flagged", "affected": int(balance_clean["BB_STATUS_DELINQUENT"].sum())},
    {"rule": "Unknown monthly statuses flagged", "affected": int(balance_clean["BB_STATUS_UNKNOWN"].sum())},
    {"rule": "Features removed by missingness/constant policy", "affected": len(removed_features)},
])
balance_clean.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "bureau_balance_feature_decisions.csv", index=False)
cleaning_audit.to_csv(audit_folder / "bureau_balance_cleaning_audit.csv", index=False)
validation_checks.to_csv(audit_folder / "bureau_balance_cleaning_validation.csv", index=False)
print("Clean bureau-balance dataset saved:", output_path)
print("Output rows:", len(balance_clean))
print("Output columns:", balance_clean.shape[1])
print("Unique bureau accounts:", balance_clean["SK_ID_BUREAU"].nunique())
print("Unique applicants:", balance_clean["SK_ID_CURR"].nunique())
print("Remaining numerical missing values:", int(balance_clean.select_dtypes(include="number").isna().sum().sum()))

Clean bureau-balance dataset saved: /Users/taranveersingh/A-MRP/data/interim/bureau_balance_clean.pkl
Output rows: 14701612
Output columns: 10
Unique bureau accounts: 523515
Unique applicants: 92231
Remaining numerical missing values: 2837039


## Main cleaning results

The cleaned bureau-balance data contains 14,701,612 monthly records for 523,515 bureau accounts and 92,231 applicants. Only records that connect to the cleaned bureau data are kept.

The status codes and month values were already valid, so no corrections were actually needed there. A few status indicator columns (delinquent, closed, unknown) were created to make this table easier to summarize later.

Two features were removed because they ended up constant. The cleaned data has 10 columns. The next step is to clean the credit-card balance data.
